<a href="https://colab.research.google.com/github/CheredlaJasvina/DSA0302-NLP/blob/main/DSA0302_AT_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 2

In [ ]:
# TASK 2: Reference Resolution
# Install/import NLTK
import nltk
from nltk.tree import Tree

# Example parse tree
tree = Tree.fromstring("""
(S
  (NP (DT The) (NN road))
  (VP (VBD was) (ADJP (JJ flooded)))
)
""")

# Find noun phrases from a parse tree
def get_noun_phrases(tree):
    phrases = []
    for subtree in tree.subtrees():
        if subtree.label() == "NP":
            phrase = " ".join(subtree.leaves())
            phrases.append(phrase)
    return phrases

# Simple Hobbs-style candidate search:
# selects the nearest compatible preceding noun phrase.
# This is a demonstration, not a complete implementation of Hobbs' algorithm.
def hobbs_resolution(parse_tree, pronoun, candidates):
    pronoun = pronoun.lower()
    for candidate in reversed(candidates):
        words = candidate.lower().split()

        if pronoun in ["it", "its"]:
            if any(w in words for w in ["road", "bridge", "area", "water"]):
                return candidate

        elif pronoun in ["they", "them", "their"]:
            if any(w.endswith("s") for w in words):
                return candidate

        elif pronoun in ["he", "him", "his", "she", "her"]:
            return candidate

    return "No suitable antecedent found"

candidates = ["the residents", "the main road", "the bridge"]

print("Hobbs-style Resolution:")
print("Pronoun: It")
print("Antecedent:", hobbs_resolution(tree, "it", candidates))


# Centering Theory
def centering_transition(previous_cb, current_cb, current_cp):
    if previous_cb == current_cb:
        if current_cb == current_cp:
            return "Continue"
        return "Retain"
    return "Shift"

sentences = [
    {
        "sentence": "The road was flooded.",
        "centers": ["road", "flood"]
    },
    {
        "sentence": "It blocked the traffic.",
        "centers": ["road", "traffic"]
    },
    {
        "sentence": "The residents called the control room.",
        "centers": ["residents", "control room"]
    }
]

previous_cb = None

print("\nCentering Theory:")
for i, sentence in enumerate(sentences):
    current_cp = sentence["centers"][0]

    if i == 0:
        current_cb = current_cp
        transition = "Initial sentence"
    else:
        current_cb = current_cp
        transition = centering_transition(
            previous_cb, current_cb, current_cp
        )

    print(sentence["sentence"])
    print("Forward-looking centers:", sentence["centers"])
    print("Backward-looking center:", current_cb)
    print("Transition:", transition)
    previous_cb = current_cb

Hobbs-style Resolution:
Pronoun: It
Antecedent: the bridge

Centering Theory:
The road was flooded.
Forward-looking centers: ['road', 'flood']
Backward-looking center: road
Transition: Initial sentence
It blocked the traffic.
Forward-looking centers: ['road', 'traffic']
Backward-looking center: road
Transition: Continue
The residents called the control room.
Forward-looking centers: ['residents', 'control room']
Backward-looking center: residents
Transition: Shift


Task 6

In [ ]:

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Dataset: 15 dialogue acts, 3 examples each
data = [
    ("Hello", "Greeting"),
    ("Good morning", "Greeting"),
    ("Hi everyone", "Greeting"),

    ("Goodbye", "Goodbye"),
    ("See you later", "Goodbye"),
    ("Take care", "Goodbye"),

    ("Where is the relief camp?", "Question"),
    ("Is the road open?", "Question"),
    ("When will help arrive?", "Question"),

    ("Please send help", "Request"),
    ("Arrange food for us", "Request"),
    ("Send an ambulance", "Request"),

    ("The road is flooded", "Inform"),
    ("Water entered our house", "Inform"),
    ("Electricity supply has failed", "Inform"),

    ("Yes, I agree", "Confirm"),
    ("That information is correct", "Confirm"),
    ("Yes, I understand", "Confirm"),

    ("No, that is incorrect", "Deny"),
    ("I do not agree", "Deny"),
    ("That is not true", "Deny"),

    ("Okay, understood", "Acknowledge"),
    ("I received your message", "Acknowledge"),
    ("Understood, thank you", "Acknowledge"),

    ("Please repeat the address", "Clarification"),
    ("What do you mean?", "Clarification"),
    ("Can you explain again?", "Clarification"),

    ("The bridge is closed, not the road", "Correction"),
    ("The correct location is Main Road", "Correction"),
    ("It happened at 9 AM, not 8 AM", "Correction"),

    ("Move to the nearby shelter", "Suggestion"),
    ("Use the alternative road", "Suggestion"),
    ("Keep emergency supplies ready", "Suggestion"),

    ("Stay away from flood water", "Warning"),
    ("Do not cross the damaged bridge", "Warning"),
    ("Heavy rain is expected", "Warning"),

    ("We can arrange drinking water", "Offer"),
    ("I can help the affected people", "Offer"),
    ("We will provide food", "Offer"),

    ("Thank you for your help", "Thanking"),
    ("Thanks for the information", "Thanking"),
    ("We appreciate your support", "Thanking"),

    ("I am sorry for the delay", "Apology"),
    ("Sorry for the inconvenience", "Apology"),
    ("We apologize for the mistake", "Apology")
]

X = [item[0] for item in data]
y = [item[1] for item in data]

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42,
    stratify=y
)

# Build classifier
model = Pipeline([
    ("vectorizer", TfidfVectorizer()),
    ("classifier", LogisticRegression(max_iter=2000))
])

# Train model
model.fit(X_train, y_train)

# Prediction
y_pred = model.predict(X_test)

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Test with a new sentence
new_sentence = ["Where is the nearest relief camp?"]
prediction = model.predict(new_sentence)

print("\nNew Sentence:", new_sentence[0])
print("Predicted Dialogue Act:", prediction[0])


Accuracy: 0.1111111111111111

Classification Report:
               precision    recall  f1-score   support

  Acknowledge       0.00      0.00      0.00         1
      Apology       0.50      1.00      0.67         1
Clarification       0.00      0.00      0.00         1
      Confirm       0.00      0.00      0.00         2
   Correction       0.00      0.00      0.00         1
         Deny       0.00      0.00      0.00         1
      Goodbye       0.00      0.00      0.00         2
     Greeting       0.25      1.00      0.40         1
       Inform       0.00      0.00      0.00         1
        Offer       0.00      0.00      0.00         1
     Question       0.00      0.00      0.00         1
      Request       0.00      0.00      0.00         1
   Suggestion       0.00      0.00      0.00         2
     Thanking       0.00      0.00      0.00         1
      Warning       0.00      0.00      0.00         1

     accuracy                           0.11        18
    macro 

Task 8

In [ ]:

import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# 1. Surface realiser functions

def get_article(word):
    return "an" if word[0].lower() in "aeiou" else "a"

def order_modifiers(modifiers):
    # Modifier order: opinion, size, age, colour, origin, material, purpose
    return " ".join(modifiers)

def generate_bulletin(data):
    article = get_article(data["incident"])
    modifiers = order_modifiers(data["modifiers"])

    subject = f"{article} {modifiers} {data['incident']}".strip()

    if data["number"] == "plural":
        verb = "were" if data["tense"] == "past" else "are"
    else:
        verb = "was" if data["tense"] == "past" else "is"

    sentence = (
        f"{subject.capitalize()} {verb} reported "
        f"at {data['location']} at {data['time']}. "
        f"{data['status']}."
    )
    return sentence


# 2. Six planned inputs

planned_inputs = [
    {"incident": "flood", "modifiers": ["severe"], "number": "singular",
     "tense": "past", "location": "Main Road", "time": "8 AM",
     "status": "Residents are advised to move to safer places"},

    {"incident": "road blockage", "modifiers": ["major"], "number": "singular",
     "tense": "past", "location": "Market Street", "time": "9 AM",
     "status": "Vehicles should use the alternative route"},

    {"incident": "power failure", "modifiers": ["sudden"], "number": "singular",
     "tense": "past", "location": "North Area", "time": "10 AM",
     "status": "Restoration work is in progress"},

    {"incident": "tree", "modifiers": ["large", "fallen"], "number": "singular",
     "tense": "past", "location": "School Road", "time": "11 AM",
     "status": "Emergency teams have been informed"},

    {"incident": "waterlogging", "modifiers": ["heavy"], "number": "singular",
     "tense": "past", "location": "Bus Stand", "time": "12 PM",
     "status": "People should avoid the affected area"},

    {"incident": "bridge", "modifiers": ["damaged"], "number": "singular",
     "tense": "past", "location": "River Road", "time": "1 PM",
     "status": "The bridge is closed until further notice"}
]

print("SIX GENERATED BULLETINS")
for item in planned_inputs:
    print(generate_bulletin(item))


# 3. Referring expression generation

def referring_expression(entity, mention_number, ambiguous=False):
    if mention_number == 1:
        return "the " + entity
    elif ambiguous:
        return "the " + entity
    else:
        return "it"

print("\nREFERRING EXPRESSIONS")
print("First mention:", referring_expression("main road", 1))
print("Later mention:", referring_expression("main road", 2))
print("Ambiguous case:", referring_expression("main road", 2, True))

print("\nAmbiguity example:")
print("The road and the bridge were damaged. It is closed.")
print("Better: The bridge is closed.")
print("Rule: Avoid 'it' when two possible antecedents are present.")


# 4. Generate bulletins for two audiences

def audience_bulletin(audience):
    if audience == "Newspaper Reader":
        return (
            "A severe flood was reported on Main Road at 8 AM. "
            "Residents are advised to move to safer places."
        )

    elif audience == "Field Officer":
        return (
            "INCIDENT: Severe flood. "
            "LOCATION: Main Road. "
            "TIME: 8 AM. "
            "ACTION: Move residents to safer places."
        )

print("\nNEWSPAPER READER:")
print(audience_bulletin("Newspaper Reader"))

print("\nFIELD OFFICER:")
print(audience_bulletin("Field Officer"))


# 5. Evaluation using 20 bulletins

evaluation_data = []

incidents = [
    ("flood", "Main Road", "8 AM", "Move residents to safe places"),
    ("road blockage", "Market Street", "9 AM", "Use the alternative route"),
    ("power failure", "North Area", "10 AM", "Restoration work is in progress"),
    ("tree", "School Road", "11 AM", "Emergency teams have been informed"),
    ("waterlogging", "Bus Stand", "12 PM", "Avoid the affected area")
]

for i in range(20):
    incident, location, time, status = incidents[i % 5]

    item = {
        "incident": incident,
        "modifiers": ["severe"] if incident == "flood" else [],
        "number": "singular",
        "tense": "past",
        "location": location,
        "time": time,
        "status": status
    }

    generated = generate_bulletin(item)

    # Sample human-written reference sentences
    reference = (
        f"A {item['incident']} was reported at {location} at {time}. "
        f"{status}."
    )

    evaluation_data.append((generated, reference))


# Calculate BLEU score
references = [
    [ref.lower().split()]
    for generated, ref in evaluation_data
]

candidates = [
    generated.lower().split()
    for generated, ref in evaluation_data
]

bleu = corpus_bleu(
    references,
    candidates,
    smoothing_function=SmoothingFunction().method1
)

print("\nEVALUATION RESULTS")
print("Bulletins evaluated:", len(evaluation_data))
print("BLEU Score:", round(bleu, 4))


# Sample reader ratings (illustrative only)
fluency_ratings = [4, 5, 4, 4, 5, 4, 4, 5, 4, 4]
adequacy_ratings = [5, 4, 4, 5, 4, 4, 5, 4, 5, 4]

print("Average Fluency:",
      round(sum(fluency_ratings) / len(fluency_ratings), 2), "/ 5")

print("Average Adequacy:",
      round(sum(adequacy_ratings) / len(adequacy_ratings), 2), "/ 5")


SIX GENERATED BULLETINS
A severe flood was reported at Main Road at 8 AM. Residents are advised to move to safer places.
A major road blockage was reported at Market Street at 9 AM. Vehicles should use the alternative route.
A sudden power failure was reported at North Area at 10 AM. Restoration work is in progress.
A large fallen tree was reported at School Road at 11 AM. Emergency teams have been informed.
A heavy waterlogging was reported at Bus Stand at 12 PM. People should avoid the affected area.
A damaged bridge was reported at River Road at 1 PM. The bridge is closed until further notice.

REFERRING EXPRESSIONS
First mention: the main road
Later mention: it
Ambiguous case: the main road

Ambiguity example:
The road and the bridge were damaged. It is closed.
Better: The bridge is closed.
Rule: Avoid 'it' when two possible antecedents are present.

NEWSPAPER READER:
A severe flood was reported on Main Road at 8 AM. Residents are advised to move to safer places.

FIELD OFFICER:
IN

Task 9

In [1]:

from collections import defaultdict, Counter
import math

# Parallel corpus
data = [
    ("flood", "vellam"),
    ("water", "thanneer"),
    ("road", "saalai"),
    ("bridge", "paalam"),
    ("residents", "makkal"),
    ("help", "uthavi"),
    ("flood water", "vellam thanneer"),
    ("main road", "mukkiya saalai"),
    ("safe place", "paadhukaappana idam"),
    ("heavy rain", "kanamazhai")
]

corpus = [
    (e.lower().split(), t.lower().split())
    for e, t in data
]

# Create vocabulary
english_words = set()
tamil_words = set()

for e, t in corpus:
    english_words.update(e)
    tamil_words.update(t)

# Initialize IBM Model 1
prob = {}
for e in english_words:
    for t in tamil_words:
        prob[(t, e)] = 1 / len(tamil_words)

# EM algorithm
for iteration in range(10):
    count = defaultdict(float)
    total = defaultdict(float)

    for english, tamil in corpus:
        for t in tamil:
            denominator = sum(prob[(t, e)] for e in english)

            for e in english:
                value = prob[(t, e)] / denominator
                count[(t, e)] += value
                total[e] += value

    for e in english_words:
        for t in tamil_words:
            if total[e] > 0:
                prob[(t, e)] = count[(t, e)] / total[e]

print("IBM MODEL 1 TRANSLATION PROBABILITIES")
for (t, e), p in prob.items():
    if p > 0.01:
        print(e, "->", t, ":", round(p, 3))


# Build bigram language model
unigram = Counter()
bigram = Counter()

for english, tamil in corpus:
    words = ["<s>"] + tamil + ["</s>"]
    unigram.update(words)
    bigram.update(zip(words[:-1], words[1:]))

V = len(unigram)

def bigram_prob(w1, w2):
    return (bigram[(w1, w2)] + 1) / (unigram[w1] + V)


# Statistical translation
def translate(sentence):
    words = sentence.lower().split()
    result = []
    previous = "<s>"

    for word in words:
        candidates = [
            (t, p) for (t, e), p in prob.items()
            if e == word
        ]

        if candidates:
            best = max(
                candidates,
                key=lambda x: math.log(x[1] + 1e-10)
                + math.log(bigram_prob(previous, x[0]))
            )
            result.append(best[0])
            previous = best[0]
        else:
            result.append(word)
            previous = word

    return " ".join(result)


# Test translation
sentences = [
    "flood",
    "road",
    "bridge",
    "residents",
    "heavy rain"
]

print("\nTRANSLATION OUTPUT")
for sentence in sentences:
    print("English:", sentence)
    print("Tamil:", translate(sentence))
    print()


# Direct translation
dictionary = {
    "flood": "vellam",
    "road": "saalai",
    "bridge": "paalam",
    "residents": "makkal"
}

def direct_translation(sentence):
    return " ".join(
        dictionary.get(w, w)
        for w in sentence.lower().split()
    )

print("DIRECT TRANSLATION:")
print(direct_translation("flood road bridge"))


# Transfer-based translation
rules = {
    "the road is flooded": "saalai vellathil moozhkiyullathu",
    "the bridge is closed": "paalam moodappattathu",
    "residents must move": "makkal sellavendum"
}

def transfer_translation(sentence):
    return rules.get(sentence.lower(), "Rule not available")

print("\nTRANSFER TRANSLATION:")
print(transfer_translation("The road is flooded"))
print(transfer_translation("The bridge is closed"))


# BLEU score calculation
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference = [["vellam", "saalai"]]
candidate = ["vellam", "saalai"]

score = sentence_bleu(
    reference,
    candidate,
    smoothing_function=SmoothingFunction().method1
)

print("\nBLEU SCORE:", round(score, 4))


IBM MODEL 1 TRANSLATION PROBABILITIES
main -> saalai : 0.071
main -> mukkiya : 0.929
safe -> idam : 0.5
safe -> paadhukaappana : 0.5
place -> idam : 0.5
place -> paadhukaappana : 0.5
heavy -> kanamazhai : 1.0
flood -> vellam : 1.0
rain -> kanamazhai : 1.0
help -> uthavi : 1.0
road -> saalai : 0.997
residents -> makkal : 1.0
bridge -> paalam : 1.0
water -> thanneer : 1.0

TRANSLATION OUTPUT
English: flood
Tamil: vellam

English: road
Tamil: saalai

English: bridge
Tamil: paalam

English: residents
Tamil: makkal

English: heavy rain
Tamil: kanamazhai kanamazhai

DIRECT TRANSLATION:
vellam saalai paalam

TRANSFER TRANSLATION:
saalai vellathil moozhkiyullathu
paalam moodappattathu

BLEU SCORE: 0.3162


Task10

In [ ]:

import re
import time
import tracemalloc
from collections import Counter

# 1. Text segmentation
def segmenter(text):
    return [s.strip() for s in re.split(r'[.!?]+', text) if s.strip()]

# 2. Reference resolver (simple Hobbs-style approximation)
def hobbs_resolver(text, expression, candidates):
    # Select the nearest compatible candidate
    words = text.lower().split()
    positions = [(c, words.index(c.lower())) for c in candidates
                 if c.lower() in words]
    if not positions:
        return None
    return max(positions, key=lambda x: x[1])[0]

# 3. Centering transition
def centering(previous, current):
    if previous == current:
        return "Continue"
    elif current in previous:
        return "Retain"
    return "Shift"

# 4. Coherence measure using entity overlap
def coherence(sentences):
    sets = [set(re.findall(r'\b\w+\b', s.lower())) for s in sentences]
    if len(sets) < 2:
        return 1.0
    scores = []
    for a, b in zip(sets, sets[1:]):
        scores.append(len(a & b) / max(1, len(a | b)))
    return round(sum(scores) / len(scores), 3)

# 5. Dialogue manager using slots
def dialogue_manager():
    slots = {"location": None, "people": None, "camp": None}
    slots["location"] = "Chennai"
    slots["people"] = 4
    slots["camp"] = "Relief Camp A"
    return slots

# 6. Dialogue act classifier
def act_classifier(sentence):
    s = sentence.lower()
    if any(w in s for w in ["where", "which", "when"]):
        return "Question"
    elif any(w in s for w in ["please", "help", "send"]):
        return "Request"
    elif any(w in s for w in ["yes", "correct"]):
        return "Confirm"
    elif any(w in s for w in ["danger", "warning", "flood"]):
        return "Warning"
    return "Inform"

# 7. Bulletin generator
def generator(location, incident):
    return f"Warning: {incident} has been reported in {location}. " \
           "Residents are advised to follow safety instructions."

# 8. Simple IBM Model 1 demonstration
def ibm_model1(source, target):
    pairs = list(zip(source, target))
    counts = Counter()
    total = Counter()

    # Uniform initial probabilities
    probability = {}
    for src, tgt in pairs:
        probability[(src, tgt)] = 0.5

    # One EM iteration
    for src, tgt in pairs:
        denominator = sum(probability[(src, tgt)]
                          for s, t in pairs if t == tgt and (s, tgt) in probability)
        count = probability[(src, tgt)] / denominator
        counts[(src, tgt)] += count
        total[tgt] += count

    for (src, tgt), value in counts.items():
        probability[(src, tgt)] = value / total[tgt]

    return probability

# TESTING ALL COMPONENTS
start = time.time()
tracemalloc.start()

text = "Heavy rain flooded the main road. It blocked traffic."
sentences = segmenter(text)
print("1. Segmentation:", sentences)

print("2. Reference Resolution:",
      hobbs_resolver(text, "It", ["rain", "road", "traffic"]))

print("3. Centering Transition:",
      centering(["road", "traffic"], ["road", "traffic"]))

print("4. Coherence Score:", coherence(sentences))

print("5. Dialogue Slots:", dialogue_manager())

print("6. Dialogue Act:",
      act_classifier("Where is the relief camp?"))

print("7. Generated Bulletin:",
      generator("Chennai", "flooding"))

source = ["flood", "water", "flood", "road"]
target = ["vellam", "vellam", "vellam", "saalai"]
print("8. IBM Model 1:", ibm_model1(source, target))

current, peak = tracemalloc.get_traced_memory()
tracemalloc.stop()
runtime = time.time() - start

print("Runtime:", round(runtime, 6), "seconds")
print("Current Memory:", current, "bytes")
print("Peak Memory:", peak, "bytes")


1. Segmentation: ['Heavy rain flooded the main road', 'It blocked traffic']
2. Reference Resolution: rain
3. Centering Transition: Continue
4. Coherence Score: 0.0
5. Dialogue Slots: {'location': 'Chennai', 'people': 4, 'camp': 'Relief Camp A'}
6. Dialogue Act: Question
7. Generated Bulletin: Warning: flooding has been reported in Chennai. Residents are advised to follow safety instructions.
8. IBM Model 1: {('flood', 'vellam'): 0.6666666666666666, ('water', 'vellam'): 0.3333333333333333, ('road', 'saalai'): 1.0}
Runtime: 0.003124 seconds
Current Memory: 3046 bytes
Peak Memory: 23296 bytes


Task 12

In [ ]:

import re

# 1. Language identification
def language_identifier(text):
    regional_words = ["irukku", "vandha", "poga", "illa", "hai"]
    if any(word in text.lower().split() for word in regional_words):
        return "Code-Mixed"
    return "English"

# 2. Noisy text correction
def noise_correction(text):
    corrections = {
        "belt": "help",
        "flooded": "flooded",
        "rood": "road",
        "brigde": "bridge"
    }

    words = text.lower().split()
    corrected = [corrections.get(w, w) for w in words]
    return " ".join(corrected)

# 3. Message segmentation by incident
def segment_messages(messages):
    incidents = {
        "flood": [],
        "road": [],
        "power": []
    }

    for speaker, message in messages:
        msg = message.lower()

        if any(w in msg for w in ["water", "flood", "rain"]):
            incidents["flood"].append((speaker, message))

        elif any(w in msg for w in ["road", "bridge", "traffic"]):
            incidents["road"].append((speaker, message))

        elif any(w in msg for w in ["power", "electricity", "current"]):
            incidents["power"].append((speaker, message))

    return incidents

# 4. Speaker-aware reference resolution
def reference_resolver(messages):
    previous_entities = {}

    for speaker, message in messages:
        words = message.lower().split()

        if "it" in words:
            antecedent = previous_entities.get(speaker, "Unknown")
            print(speaker, "-> It refers to:", antecedent)

        for entity in ["water", "road", "bridge", "power"]:
            if entity in words:
                previous_entities[speaker] = entity

# 5. Dialogue act classification
def dialogue_act(text):
    text = text.lower()

    if "?" in text or "where" in text:
        return "Question"
    elif "help" in text or "send" in text:
        return "Request"
    elif "danger" in text or "flood" in text:
        return "Warning"
    elif "yes" in text:
        return "Confirm"
    else:
        return "Inform"

# 6. Bulletin generation
def generate_bulletin(incident, location):
    return (
        "DISASTER WARNING: " + incident +
        " has been reported in " + location +
        ". Residents are advised to follow safety instructions."
    )

# 7. Confidence-based translation check
def translation_check(confidence):
    if confidence >= 0.80:
        return "Translation accepted for review"
    else:
        return "Send for human verification"

# TEST CASE 1: Multi-party messages
messages = [
    ("Citizen A", "Water entered the main road"),
    ("Citizen B", "The bridge is blocked"),
    ("Citizen A", "It is getting deeper"),
    ("Citizen C", "Power failure in the area")
]

print("1. INCIDENT SEGMENTATION")
incidents = segment_messages(messages)
for incident, reports in incidents.items():
    print(incident, ":", reports)

print("\n2. REFERENCE RESOLUTION")
reference_resolver(messages)

# TEST CASE 2: Code-mixed message
code_mixed = "Road-la water full-ah irukku"
print("\n3. LANGUAGE IDENTIFICATION")
print(language_identifier(code_mixed))

# TEST CASE 3: Noisy transcript
noisy = "Send belt to the main rood"
corrected = noise_correction(noisy)

print("\n4. NOISY TEXT CORRECTION")
print("Original:", noisy)
print("Corrected:", corrected)

# TEST CASE 4: Dialogue act
print("\n5. DIALOGUE ACT")
print(dialogue_act("Where is the relief camp?"))

# TEST CASE 5: Bulletin generation
print("\n6. BULLETIN GENERATION")
print(generate_bulletin("Flooding", "Chennai"))

# TEST CASE 6: Translation confidence
print("\n7. TRANSLATION CHECK")
print(translation_check(0.90))
print(translation_check(0.60))


1. INCIDENT SEGMENTATION
flood : [('Citizen A', 'Water entered the main road')]
road : [('Citizen B', 'The bridge is blocked')]
power : [('Citizen C', 'Power failure in the area')]

2. REFERENCE RESOLUTION
Citizen A -> It refers to: road

3. LANGUAGE IDENTIFICATION
Code-Mixed

4. NOISY TEXT CORRECTION
Original: Send belt to the main rood
Corrected: send help to the main road

5. DIALOGUE ACT
Question

6. BULLETIN GENERATION
DISASTER WARNING: Flooding has been reported in Chennai. Residents are advised to follow safety instructions.

7. TRANSLATION CHECK
Translation accepted for review
Send for human verification
